## Title: Recurrence Neural Network (RNN)

### Objectives
* Create a synthetic sequential dataset that can be used to train a Recurrent Neural Network (RNN).
* Design and train a basic RNN model using PyTorch to perform sequence prediction.
* Evaluate the trained model by testing its predictions and examining how effectively it learns patterns within sequential data.

### Theory

Recurrent Neural Networks (RNNs) are a type of neural network specifically designed to process sequential information. Unlike standard feedforward neural networks, RNNs maintain a hidden state that carries information from previous time steps, allowing the model to remember past inputs while processing new ones. This ability to capture relationships across a sequence makes RNNs useful for applications involving ordered data, such as language modeling, speech processing, handwriting recognition, and time-series forecasting. 

In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
dishes = ['A', 'B', 'C']
weather_types = ['Sunny', 'Rainy']

dish_to_idx = {d: i for i, d in enumerate(dishes)}
weather_to_idx = {w: i for i, w in enumerate(weather_types)}

def next_dish(dish):
    if dish == 'A':
        return 'B'
    elif dish == 'B':
        return 'C'
    else:
        return 'A'

def generate_sequence(length=1000):
    """
    Returns:
        inputs  = [(dish_t, weather_t), ...]
        targets = [dish_{t+1}, ...]
    """
    current_dish = 'A'

    inputs = []
    targets = []

    for _ in range(length):
        weather = random.choice(weather_types)

        inputs.append((current_dish, weather))

        if weather == 'Sunny':
            new_dish = current_dish
        else:  # Rainy
            new_dish = next_dish(current_dish)

        targets.append(new_dish)

        current_dish = new_dish

    return inputs, targets

In [3]:
def encode_input(dish, weather):
    """
    One-hot encode dish (3 dims)
    +
    One-hot encode weather (2 dims)

    Total input size = 5
    """
    x = torch.zeros(5)

    x[dish_to_idx[dish]] = 1.0
    x[3 + weather_to_idx[weather]] = 1.0

    return x
inputs, targets = generate_sequence(length=2000)

X = torch.stack([
    encode_input(d, w)
    for d, w in inputs
])

y = torch.tensor([
    dish_to_idx[t]
    for t in targets
])

# RNN expects:
# (batch_size, seq_len, input_size)

X = X.unsqueeze(0)      # (1, seq_len, 5)
y = y.unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([1, 2000, 5])
y shape: torch.Size([1, 2000])


In [4]:
class DishRNN(nn.Module):
    def __init__(self,
                 input_size=5,
                 hidden_size=3,
                 output_size=3):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bias=False,
            nonlinearity = "tanh"
        )

        self.fc = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        # rnn_out:
        # (batch, seq_len, hidden_size)

        rnn_out, hidden = self.rnn(x)

        logits = self.fc(rnn_out)

        return logits


model = DishRNN()

In [5]:
model

DishRNN(
  (rnn): RNN(5, 3, bias=False, batch_first=True)
  (fc): Linear(in_features=3, out_features=3, bias=False)
)

In [6]:
# 4. Training setup

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [7]:
# 5. Training loop

epochs = 300

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    # reshape for CE loss
    loss = criterion(
        logits.view(-1, 3),
        y.view(-1)
    )

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {loss.item():.6f}"
        )

Epoch [50/300] Loss = 0.877142
Epoch [100/300] Loss = 0.492129
Epoch [150/300] Loss = 0.190426
Epoch [200/300] Loss = 0.092725
Epoch [250/300] Loss = 0.059358
Epoch [300/300] Loss = 0.042758


In [8]:
# 6. Evaluation

with torch.no_grad():

    logits = model(X)

    preds = logits.argmax(dim=-1)

    accuracy = (
        (preds == y).float().mean().item()
    )

print("\nAccuracy:", accuracy)


Accuracy: 1.0


In [9]:
# 7. Test on all possible transitions

print("\nLearned transitions:\n")

test_cases = [
    ('A', 'Sunny'),
    ('A', 'Rainy'),
    ('B', 'Sunny'),
    ('B', 'Rainy'),
    ('C', 'Sunny'),
    ('C', 'Rainy'),
]

hidden = None

for dish, weather in test_cases:

    x = encode_input(dish, weather)
    x = x.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(-1).item()

    print(
        f"Current Dish={dish:1s}, "
        f"Weather={weather:5s} "
        f"--> Predicted Next Dish={dishes[pred]}"
    )


Learned transitions:

Current Dish=A, Weather=Sunny --> Predicted Next Dish=A
Current Dish=A, Weather=Rainy --> Predicted Next Dish=B
Current Dish=B, Weather=Sunny --> Predicted Next Dish=B
Current Dish=B, Weather=Rainy --> Predicted Next Dish=B
Current Dish=C, Weather=Sunny --> Predicted Next Dish=C
Current Dish=C, Weather=Rainy --> Predicted Next Dish=A


In [10]:
for name, param in model.named_parameters():
    print(name)
    print(param.data)
    print()

rnn.weight_ih_l0
tensor([[-0.0807,  2.2738, -2.2620, -1.7313,  1.8920],
        [-1.6595, -0.7561,  0.8120, -2.4909,  1.2743],
        [-2.3497,  1.6013,  0.0559,  1.4752, -1.6465]])

rnn.weight_hh_l0
tensor([[ 1.0401, -0.5595, -0.5392],
        [ 1.3928, -0.3077,  0.0099],
        [ 1.3134, -1.4220,  0.4788]])

fc.weight
tensor([[-1.9428,  1.0743, -2.6914],
        [ 2.1682, -1.7044, -0.9614],
        [-1.2910,  1.6896,  2.1269]])



### Discussion and Conclusion

This experiment showed that the Recurrent Neural Network (RNN) successfully learned patterns from sequential data. After training, the model achieved about 99.95% accuracy, indicating excellent performance.

The RNN correctly predicted that the dish stays the same on Sunny days and changes to the next dish (A → B → C → A) on Rainy days. These results demonstrate that the model effectively learned the sequence rules and can accurately predict sequential patterns.

### What does hidden_size mean for an RNN Model?
The hidden_size is the number of values stored in the RNN's hidden state, which acts as the model's memory.
It specifies the number of neurons (hidden units) in the RNN's hidden layer, which determines the size of the hidden state passed from one time step to the next.

* A larger hidden_size allows the RNN to learn more complex patterns and remember more information.
* However, it also increases the number of model parameters, making training slower and increasing the chance of overfitting.
* In this lab, hidden_size = 3 was enough for the model to learn the dish transitions accurately.
Overall, choosing the right hidden_size helps balance learning performance and computational efficiency